## Auto Reload


In [1]:
%load_ext autoreload
%autoreload 2

## Run Simuation

In [4]:
import numpy as np
import plotly.graph_objects as go

from dynamics import DroneDynamics
from guidance import NominalGuidance
from safety_filter import SafetyFilter
import scenarios


def run_simulation(scenario_name="chase", animate=True):
    # 1. Load Scenario
    if scenario_name not in scenarios.SCENARIOS:
        print(f"Error: Scenario '{scenario_name}' not found.")
        return
    mission = scenarios.SCENARIOS[scenario_name]()
    print(f"Loaded: {mission.name} | Animate: {animate}")

    # 2. Setup
    dt = 0.01
    drone_physics = DroneDynamics(dt=dt)
    guidance = NominalGuidance()
    safety_filter = SafetyFilter(u_max=10.0)
    current_state = mission.start_state

    # 3. Data Storage
    path_history = []
    target_history = []
    obs_history = []

    total_steps = int(mission.duration / dt)

    # 4. Simulation Loop
    print(f"Simulating {total_steps} steps...")
    for t_step in range(total_steps):
        time = t_step * dt

        # A. Update Scenario (Target + Obstacles)
        target_pos, target_vel = mission.target.update(time)

        current_obs_snapshot = []
        for obs in mission.obstacles:
            if np.linalg.norm(obs.velocity) > 0:
                obs.center += obs.velocity * dt
                # Simple bounce logic
                if abs(obs.center[1]) > 4.0:
                    obs.velocity[1] *= -1
            current_obs_snapshot.append(obs.center.copy())

        obs_history.append(current_obs_snapshot)

        # B. Control & Physics
        u_nom = guidance.compute_u_nom(current_state.pos, current_state.vel, target_pos, target_vel)
        u_safe = safety_filter.filter(u_nom, current_state.pos, current_state.vel, mission.obstacles)
        current_state = drone_physics.step(current_state, u_safe)

        path_history.append(current_state.pos)
        target_history.append(target_pos)

    # 5. Visualization Selector
    path_history = np.array(path_history)
    target_history = np.array(target_history)

    if animate:
        print("Generating Plotly Animation...")
        animate_plotly(path_history, target_history, obs_history, mission, dt)
    else:
        print("Plotting Static Graph...")
        # You can keep your old static plotter or use a plotly static one
        animate_plotly(path_history, target_history, obs_history, mission, dt, static_only=True)




def animate_plotly(path, target_path, obs_history, mission, dt, static_only=False):
    
    total_frames = len(path)
    # Stride: Skip frames to keep animation smooth (approx 150 frames total)
    stride = max(1, total_frames // 150) 
    
    fig = go.Figure()

    # --- HELPER: Sphere Geometry ---
    def get_sphere_data(center, radius):
        u = np.linspace(0, 2 * np.pi, 15)
        v = np.linspace(0, np.pi, 10)
        x = center[0] + radius * np.outer(np.cos(u), np.sin(v))
        y = center[1] + radius * np.outer(np.sin(u), np.sin(v))
        z = center[2] + radius * np.outer(np.ones(np.size(u)), np.cos(v))
        return x, y, z

    # --- STATIC TRACES (Background) ---
    fig.add_trace(go.Scatter3d(
        x=path[:, 0], y=path[:, 1], z=path[:, 2],
        mode='lines', line=dict(color='blue', width=4),
        name='Drone Path', opacity=0.2
    ))

    fig.add_trace(go.Scatter3d(
        x=target_path[:, 0], y=target_path[:, 1], z=target_path[:, 2],
        mode='lines', line=dict(color='green', width=4, dash='dash'),
        name='Target Path', opacity=0.2
    ))

    # --- DYNAMIC TRACES (Actors) ---
    fig.add_trace(go.Scatter3d(
        x=[path[0, 0]], y=[path[0, 1]], z=[path[0, 2]],
        mode='markers', marker=dict(color='blue', size=6),
        name='Drone'
    ))

    fig.add_trace(go.Scatter3d(
        x=[target_path[0, 0]], y=[target_path[0, 1]], z=[target_path[0, 2]],
        mode='markers', marker=dict(color='green', size=8, symbol='diamond'),
        name='Target'
    ))

    obs_start = obs_history[0]
    for i, obs in enumerate(mission.obstacles):
        x_s, y_s, z_s = get_sphere_data(obs_start[i], obs.radius)
        fig.add_trace(go.Surface(
            x=x_s, y=y_s, z=z_s,
            colorscale=[[0, 'red'], [1, 'red']], showscale=False, opacity=0.5,
            name=f'Obs {i}'
        ))

    # --- ANIMATION FRAMES ---
    frames = []
    num_obstacles = len(mission.obstacles)
    dynamic_indices = [2, 3] + list(range(4, 4 + num_obstacles))

    for k in range(0, total_frames, stride):
        frame_data = []
        
        # 1. Update Positions
        drone_pos = path[k]
        target_pos = target_path[k]
        
        frame_data.append(go.Scatter3d(x=[drone_pos[0]], y=[drone_pos[1]], z=[drone_pos[2]])) # Drone
        frame_data.append(go.Scatter3d(x=[target_pos[0]], y=[target_pos[1]], z=[target_pos[2]])) # Target
        
        current_obs = obs_history[k]
        for i in range(num_obstacles):
            x_new, y_new, z_new = get_sphere_data(current_obs[i], mission.obstacles[i].radius)
            frame_data.append(go.Surface(x=x_new, y=y_new, z=z_new))

        # 2. CALCULATE DYNAMIC CAMERA
        # Find the midpoint between drone and target
        mid_x = (drone_pos[0] + target_pos[0]) / 2
        mid_y = (drone_pos[1] + target_pos[1]) / 2
        mid_z = (drone_pos[2] + target_pos[2]) / 2
        
        # Determine "Zoom Level" (Distance between them + Padding)
        dist = np.linalg.norm(drone_pos - target_pos)
        window = max(10.0, dist * 1.5) # Ensure window is at least 10 meters wide
        half_win = window / 2

        # Create the dynamic layout for THIS specific frame
        frame_layout = dict(
            scene=dict(
                xaxis=dict(range=[mid_x - half_win, mid_x + half_win]),
                yaxis=dict(range=[mid_y - half_win, mid_y + half_win]),
                zaxis=dict(range=[mid_z - half_win, mid_z + half_win])
            )
        )

        frames.append(go.Frame(data=frame_data, layout=frame_layout, traces=dynamic_indices, name=str(k)))

    fig.frames = frames

    # --- INITIAL LAYOUT ---
    # We set the initial view to match the first frame's calculation
    start_mid = (path[0] + target_path[0]) / 2
    fig.update_layout(
        width=1000, height=800,
        title=f"Sim: {mission.name} (Follow Camera)",
        scene=dict(
            xaxis=dict(range=[start_mid[0]-10, start_mid[0]+10], title="X"),
            yaxis=dict(range=[start_mid[1]-10, start_mid[1]+10], title="Y"),
            zaxis=dict(range=[start_mid[2]-10, start_mid[2]+10], title="Z"),
            aspectmode='cube'
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        updatemenus=[dict(
            type="buttons",
            buttons=[dict(label="Play",
                          method="animate",
                          args=[None, dict(frame=dict(duration=20, redraw=True), 
                                           fromcurrent=True)])]
        )]
    )

    if static_only:
        fig.show()
    else:
        fig.show()


if __name__ == "__main__":
    run_simulation("head_on", animate=True)
    #run_simulation("chase", animate=True)
    #run_simulation("clutter", animate=True)

Loaded: Head On Collision Test | Animate: True
Simulating 1500 steps...
Generating Plotly Animation...
